In [ ]:
%pip install pandas

In [ ]:
BUCKET = "dark-data-discovery-fsl-data"
!mkdir -p /home/jupyter/data_cache/saml_d_source
!rm -f /home/jupyter/data_cache/saml_d_source/archive.zip
!gcloud storage cp gs://{BUCKET}/sources/saml_d/SAML-D.csv /home/jupyter/data_cache/saml_d_source/
!ls -lh /home/jupyter/data_cache/saml_d_source/

In [ ]:
import pandas as pd
from pathlib import Path

SRC = next(Path.home().joinpath("data_cache/saml_d_source").glob("*.csv"))
print(SRC, f"{SRC.stat().st_size/1e9:.2f} GB")
cols = pd.read_csv(SRC, nrows=0).columns.tolist()
print(cols)
print(pd.read_csv(SRC, nrows=5).to_string())

lab_col = [c for c in cols if "laund" in c.lower() and "type" in c.lower()][0]
vc = pd.read_csv(SRC, usecols=[lab_col], dtype="category")[lab_col].value_counts()
print(f"\n{lab_col}: {vc.size} klas, {vc.sum():,} wierszy")
print(vc.to_string())

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

SRC = "/home/jupyter/data_cache/saml_d_source/SAML-D.csv"
MIN_SUSP, MIN_WIN, K, Q, NWAY, EPISODES = 2, 10, 5, 5, 5, 400

dt = {'Payment_currency':'category','Received_currency':'category','Sender_bank_location':'category',
      'Receiver_bank_location':'category','Payment_type':'category','Laundering_type':'category',
      'Is_laundering':'uint8','Amount':'float32','Sender_account':'int64','Receiver_account':'int64'}
df = pd.read_csv(SRC, dtype=dt)
print("wierszy:", len(df), "| daty:", df['Date'].min(), "->", df['Date'].max())

d = pd.to_datetime(df['Date'])
df['day'] = (d - d.min()).dt.days.astype('int32')
df['week'] = (df['day'] // 7).astype('int32')
hh = pd.to_numeric(df['Time'].str[:2], errors='coerce')
df['night'] = (hh < 6).fillna(False).astype('uint8')

def diff_flag(a, b):
    cs = df[a].cat.categories.union(df[b].cat.categories)
    return (df[a].cat.set_categories(cs).cat.codes != df[b].cat.set_categories(cs).cat.codes).astype('uint8')
df['xborder'] = diff_flag('Sender_bank_location','Receiver_bank_location')
df['curm']    = diff_flag('Payment_currency','Received_currency')
df['cash'] = df['Payment_type'].astype(str).str.contains('cash', case=False).astype('uint8')
df['r100'] = (df['Amount'] % 100 == 0).astype('uint8')
df['s10k'] = df['Amount'].between(9000, 10000).astype('uint8')
pt_cols = []
for t in df['Payment_type'].value_counts().index[:6]:
    k = 'pt_' + str(t).lower().replace(' ','_').replace('-','_')
    df[k] = (df['Payment_type'] == t).astype('uint8'); pt_cols.append(k)

base = ['Amount','week','day','night','xborder','curm','cash','r100','s10k',
        'Is_laundering','Laundering_type'] + pt_cols
o = df[['Sender_account','Receiver_account']+base].rename(
        columns={'Sender_account':'acc','Receiver_account':'cpty'}); o['dir'] = np.uint8(1)
i = df[['Receiver_account','Sender_account']+base].rename(
        columns={'Receiver_account':'acc','Sender_account':'cpty'}); i['dir'] = np.uint8(0)
lf = pd.concat([o, i], ignore_index=True, copy=False); del o, i, df
lf['amt_out'] = (lf['Amount'] * lf['dir']).astype('float32')
lf['amt_in']  = (lf['Amount'] * (1 - lf['dir'])).astype('float32')
print("wierszy long-form:", len(lf))

W = lf.groupby(['acc','week'], observed=True, sort=False).agg(
    n=('dir','size'), n_out=('dir','sum'),
    amt_mean=('Amount','mean'), amt_std=('Amount','std'),
    amt_max=('Amount','max'), amt_min=('Amount','min'),
    amt_out_sum=('amt_out','sum'), amt_in_sum=('amt_in','sum'),
    cpty_u=('cpty','nunique'), n_days=('day','nunique'),
    xb=('xborder','mean'), curm=('curm','mean'), cash=('cash','mean'),
    night=('night','mean'), r100=('r100','mean'), s10k=('s10k','mean'),
    **{k:(k,'mean') for k in pt_cols})
W['amt_std'] = W['amt_std'].fillna(0.0)
W['out_ratio']  = W['n_out'] / W['n']
W['net']        = W['amt_out_sum'] - W['amt_in_sum']
W['flow_ratio'] = np.log1p(W['amt_out_sum']) - np.log1p(W['amt_in_sum'])
W['tx_per_day'] = W['n'] / W['n_days']
print("okien:", len(W))

def dominant(sub):
    c = sub.groupby(['acc','week','Laundering_type'], observed=True).size().reset_index(name='k')
    return c.sort_values('k').drop_duplicates(['acc','week'], keep='last').set_index(['acc','week'])
S = dominant(lf[lf['Is_laundering'] == 1]); N = dominant(lf[lf['Is_laundering'] == 0])
del lf

for m in (1, 2):
    t = S[S['k'] >= m]['Laundering_type'].astype(str).value_counts()
    print(f"\n-- okna podejrzane, MIN_SUSP={m} (klas: {t.size}, ≥{MIN_WIN} okien: {(t>=MIN_WIN).sum()})")
    print(t.to_string())

lab = pd.Series(index=W.index, dtype=object)
lab.update(N['Laundering_type'].astype(str))
lab.update(S[S['k'] >= MIN_SUSP]['Laundering_type'].astype(str))
W['label'] = lab; W = W.dropna(subset=['label'])

vc = W['label'].value_counts()
print("\n-- okna per klasa (wszystkie):"); print(vc.to_string())
classes = vc[vc >= MIN_WIN].index.to_numpy()
susp_all = set(S['Laundering_type'].astype(str).unique())
print(f"\nklas ≥{MIN_WIN} okien: {len(classes)} (podejrzanych: {sum(c in susp_all for c in classes)})")

rng = np.random.default_rng(0)
perm = rng.permutation(classes)
stat_cls, ep_cls = perm[:len(perm)//2], perm[len(perm)//2:]
print(f"klasy statystyk: {len(stat_cls)} | klasy epizodów: {len(ep_cls)}")

feat = [c for c in W.columns if c != 'label']
X = W[feat].astype('float64')
heavy = [c for c in feat if X[c].min() >= 0 and X[c].max() > 50]
X[heavy] = np.log1p(X[heavy])
m_ = W['label'].isin(stat_cls)
mu, sd = X[m_].mean(), X[m_].std().replace(0, 1.0)
Z = ((X - mu) / sd).to_numpy()
by = {c: Z[(W['label'] == c).to_numpy()] for c in classes}

def nc_test(pool, tag):
    if len(pool) < NWAY:
        print(f"{tag}: za mało klas ({len(pool)}), pominięte"); return
    a = []
    for _ in range(EPISODES):
        pick = rng.choice(len(pool), NWAY, replace=False)
        P, Qs, qy = [], [], []
        for j, ci in enumerate(pick):
            A = by[pool[ci]]
            idx = rng.choice(len(A), K + Q, replace=False)
            P.append(A[idx[:K]].mean(0)); Qs.append(A[idx[K:]]); qy += [j] * Q
        P, Qm, qy = np.stack(P), np.vstack(Qs), np.array(qy)
        dd = ((Qm[:, None, :] - P[None, :, :]) ** 2).sum(-1)
        a.append(float((dd.argmin(1) == qy).mean()))
    a = np.array(a)
    print(f"{tag}: {a.mean():.3f} ±{1.96*a.std()/np.sqrt(len(a)):.3f}  (szansa {1/NWAY:.3f})")

nc_test(ep_cls, f"NC {NWAY}-way {K}-shot, wszystkie klasy epizodów")
susp_ep = np.array([c for c in classes if c in susp_all])
nc_test(susp_ep, f"NC {NWAY}-way {K}-shot, TYLKO podejrzane ({len(susp_ep)} klas)")

out = Path.home() / "data_cache/saml_d_windows_v0.parquet"
W.to_parquet(out); print("\nokna zapisane:", out, W.shape)

In [ ]:
%pip install -q pyarrow
from pathlib import Path
out = Path.home() / "data_cache/saml_d_windows_v0.parquet"
W.to_parquet(out); print("okna zapisane:", out, W.shape)

In [ ]:
import numpy as np
idx = W.index.to_frame(index=False)      # kolumny: acc, week
lab = W['label'].to_numpy()
acc = idx['acc'].to_numpy()

for c in susp_ep:
    m = lab == c
    print(f"{c:22s} okien {m.sum():5d} | kont {len(np.unique(acc[m])):4d}")

def nc_test_grouped(pool, tag, episodes=400):
    ok = [c for c in pool if len(np.unique(acc[lab == c])) >= 4]
    if len(ok) < NWAY:
        print(f"{tag}: za mało klas z ≥4 kontami ({len(ok)})"); return
    a = []
    for _ in range(episodes):
        pick = rng.choice(len(ok), NWAY, replace=False)
        P, Qs, qy = [], [], []
        for j, ci in enumerate(pick):
            m = lab == ok[ci]
            Zc, ac = Z[m], acc[m]
            ua = np.unique(ac); rng.shuffle(ua)
            half = max(1, len(ua) // 2)
            s = Zc[np.isin(ac, ua[:half])]; q = Zc[np.isin(ac, ua[half:])]
            if len(s) < K or len(q) < 1: break
            P.append(s[rng.choice(len(s), K, replace=False)].mean(0))
            nq = min(Q, len(q))
            Qs.append(q[rng.choice(len(q), nq, replace=False)]); qy += [j] * nq
        if len(P) < NWAY: continue
        P, Qm, qy = np.stack(P), np.vstack(Qs), np.array(qy)
        dd = ((Qm[:, None, :] - P[None, :, :]) ** 2).sum(-1)
        a.append(float((dd.argmin(1) == qy).mean()))
    a = np.array(a)
    print(f"{tag}: {a.mean():.3f} ±{1.96*a.std()/np.sqrt(len(a)):.3f}  (n={len(a)} epizodów)")

nc_test_grouped(susp_ep, "NC grouped (konta rozłączne), tylko podejrzane")

In [ ]:
import sys, logging, torch, torch.nn as nn
sys.path.insert(0, "/home/jupyter/mlops-meta-learning/fsl-projekt/src")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

from fsl.config import TrainConfig
from fsl.data.registry import get_loaders
from fsl.models.protonet import fast_adapt

cfg = TrainConfig(project="dark-data-discovery", dataset="samld",
                  n_way=5, k_shot=5, query=5, seed=0)
load_frozen, build = get_loaders(cfg)
bundle, sha = load_frozen(cfg)
tr, va, te = build(bundle, cfg)

d, l = tr.sample()
print("batch:", tuple(d.shape), "| etykiety:", l[:7].tolist(), "| sha:", sha[:12])

n_feat = d.shape[1]
mlp = nn.Sequential(nn.Linear(n_feat, 64), nn.LayerNorm(64), nn.ReLU(), nn.Linear(64, 64))
loss, acc = fast_adapt(mlp, (d, l), cfg.n_way, cfg.k_shot, cfg.query, torch.device("cpu"))
print("loss:", float(loss), "| acc (losowy MLP):", float(acc))

In [ ]:
import numpy as np, torch, torch.nn as nn
from fsl.models.protonet import fast_adapt

def floor(tasks, seeds=(0,1,2), episodes=200):
    out = []
    for s in seeds:
        torch.manual_seed(s)
        m = nn.Sequential(nn.Linear(26, 64), nn.LayerNorm(64), nn.ReLU(), nn.Linear(64, 64))
        m.eval()
        with torch.no_grad():
            a = [float(fast_adapt(m, tasks.sample(), cfg.n_way, cfg.k_shot,
                                  cfg.query, torch.device("cpu"))[1])
                 for _ in range(episodes)]
        out.append(np.mean(a))
        print(f"seed {s}: {np.mean(a):.4f} ±{1.96*np.std(a)/np.sqrt(len(a)):.4f}")
    print(f"podłoga (średnia z ziaren): {np.mean(out):.4f}")

floor(va)   # walidacja folda 0 — to na tym liczy się próg
floor(te)   # test, tylko do wiedzy; progu z niego nie wyprowadzaj

In [ ]:
from fsl.models.registry import get_encoder
cfg.encoder = "mlp"; cfg.in_channels = 26; cfg.image_size = 1
m = get_encoder(cfg)
d, l = tr.sample()
print(m(d).shape, "|", sum(p.numel() for p in m.parameters()), "parametrów")

In [ ]:
import torch, numpy as np
from fsl.models.protonet import fast_adapt
from fsl.models.registry import get_encoder

cfg.encoder = "mlp"; cfg.in_channels = 26; cfg.image_size = 1
torch.manual_seed(0)
m = get_encoder(cfg)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
dev = torch.device("cpu")

for it in range(1, 601):
    m.train()
    loss, acc = fast_adapt(m, tr.sample(), cfg.n_way, cfg.k_shot, cfg.query, dev)
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 100 == 0:
        m.eval()
        with torch.no_grad():
            v = [float(fast_adapt(m, va.sample(), cfg.n_way, cfg.k_shot, cfg.query, dev)[1])
                 for _ in range(200)]
        print(f"iter {it}: train_acc {float(acc):.3f} | val {np.mean(v):.4f} "
              f"±{1.96*np.std(v)/np.sqrt(len(v)):.4f}")

In [ ]:
from fsl.config import TrainConfig
c = TrainConfig(project="dark-data-discovery", dataset="resisc45",
                encoder="conv4", in_channels=3, image_size=84)
p = c.as_params()
print(len(p), p["encoder"], [k for k in p if k.startswith("mlp")])

In [ ]:
import fsl.config, inspect
print(inspect.getsourcefile(fsl.config))
print("fold" in fsl.config.TrainConfig.__dataclass_fields__)

In [ ]:
from types import SimpleNamespace
from fsl.models.registry import get_encoder

for arch in [
    {"architecture": "Conv4", "in_channels": 3, "embedding_hid": 64},
    {"encoder": "mlp", "in_channels": 26, "embedding_hid": 64, "mlp_depth": 2, "mlp_dropout": 0.0},
]:
    enc = arch.get("encoder", arch.get("architecture", "conv4")).strip().lower()
    m = get_encoder(SimpleNamespace(encoder=enc, in_channels=arch["in_channels"],
                                    embedding_hid=arch["embedding_hid"],
                                    mlp_depth=arch.get("mlp_depth", 2),
                                    mlp_dropout=arch.get("mlp_dropout", 0.0)))
    print(enc, "->", type(m).__name__)